In [1]:
# Install dependencies
!pip install -q sentence-transformers rank-bm25 groq

# Imports
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from groq import Groq
from dotenv import load_dotenv
import os, pathlib, getpass
import warnings

warnings.filterwarnings("ignore")

In [2]:
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API Key not found — enter it now: ")
print(f"GROQ   key loaded: {'yes' if os.getenv('GROQ_API_KEY') else 'NO — check unit 2/.env'}")

Groq API Key not found — enter it now:  ········


GROQ   key loaded: yes


In [3]:
import random
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [4]:
corpus = [
    "Transformers use self-attention mechanisms to encode relationships between words in a sequence.",
    "Self-attention computes pairwise interactions between tokens to capture contextual meaning.",
    "The attention mechanism assigns weights to tokens based on relevance to a query.",
    
    "Gradient descent is an optimization algorithm used to minimize loss functions.",
    "Adam optimizer combines momentum and adaptive learning rates for efficient training.",
    "Backpropagation computes gradients of loss with respect to neural network weights.",
    
    "Overfitting occurs when a model memorizes training data and fails to generalize.",
    "Regularization techniques like dropout help prevent overfitting in neural networks.",
    
    "BERT is a transformer-based model pretrained using masked language modeling.",
    "Tokenization converts raw text into tokens for model input processing.",
    
    "TF-IDF stands for Term Frequency-Inverse Document Frequency and is used in information retrieval."
]

In [29]:
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k
        
        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        
        self.sbert = SentenceTransformer('all-MiniLM-L6-v2')
        self.embeddings = self.sbert.encode(corpus, convert_to_tensor=True)

    def retrieve(self, query: str, top_k: int = 5):
        tokenized_query = query.lower().split()
        
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranks = np.argsort(bm25_scores)[::-1]
        
        query_emb = self.sbert.encode(query, convert_to_tensor=True)
        sbert_scores = util.cos_sim(query_emb, self.embeddings)[0].cpu().numpy()
        sbert_ranks = np.argsort(sbert_scores)[::-1]
        
        scores = {}
        
        for rank, doc_id in enumerate(bm25_ranks):
            scores.setdefault(doc_id, {"bm25_rank": None, "sbert_rank": None})
            scores[doc_id]["bm25_rank"] = rank + 1
        
        for rank, doc_id in enumerate(sbert_ranks):
            scores.setdefault(doc_id, {"bm25_rank": None, "sbert_rank": None})
            scores[doc_id]["sbert_rank"] = rank + 1
        
        results = []
        
        for doc_id, ranks in scores.items():
            rrf = 0
            
            if ranks["bm25_rank"]:
                rrf += 1 / (self.k + ranks["bm25_rank"])
            if ranks["sbert_rank"]:
                rrf += 1 / (self.k + ranks["sbert_rank"])
            
            results.append({
                "doc_id": doc_id,
                "rrf_score": rrf,
                "bm25_rank": ranks["bm25_rank"],
                "sbert_rank": ranks["sbert_rank"],
                "text": self.corpus[doc_id]
            })
        
        return sorted(results, key=lambda x: x["rrf_score"], reverse=True)[:top_k]

In [30]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, candidates, top_k=3):
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = cross_encoder.predict(pairs)
    
    for i, doc in enumerate(candidates):
        doc["cross_score"] = scores[i]
    
    reranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)
    
    return [
        {
            "doc_id": d["doc_id"],
            "text": d["text"],
            "cross_score": d["cross_score"]
        }
        for d in reranked[:top_k]
    ]

In [31]:
def expand_query(query):
    from groq import Groq
    import os
    
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
    
    prompt = f"""
    Generate 3 different paraphrases of this query:
    {query}
    Return each query on a new line. No numbering.
    """
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    
    text = response.choices[0].message.content
    
    queries = [q.strip() for q in text.split("\n") if q.strip()]
    
    # Ordered deduplication
    queries = list(dict.fromkeys(queries))
    
    return queries[:3]

In [14]:
retriever = HybridRetriever(corpus)

def advanced_rag(user_query: str):
    from groq import Groq
    import os
    
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
    
    expanded_queries = expand_query(user_query)
    
    all_docs = []
    seen = set()
    
    for q in expanded_queries:
        docs = retriever.retrieve(q, top_k=5)
        
        for d in docs:
            if d["text"] not in seen:
                seen.add(d["text"])
                all_docs.append(d)
    
    top_docs = rerank(user_query, all_docs, top_k=3)
    
    context = "\n".join([doc["text"] for doc in top_docs])
    
    prompt = f"""
    Answer the question using the context below.
    
    Context:
    {context}
    
    Question:
    {user_query}
    """
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    
    return response.choices[0].message.content

In [32]:
naive_model = SentenceTransformer('all-MiniLM-L6-v2')
naive_embeddings = naive_model.encode(corpus, convert_to_tensor=True)

def naive_rag(query):
    query_emb = naive_model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, naive_embeddings)[0]
    
    return corpus[scores.argmax().item()]

In [33]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is tf-idf used for?"
]

for q in queries:
    print("\nQUERY:", q)
    print("Advanced RAG Answer:\n", advanced_rag(q))


QUERY: how do transformers encode meaning?
Advanced RAG Answer:
 Transformers encode meaning by using self-attention mechanisms to compute pairwise interactions between tokens, capturing contextual relationships between words in a sequence. This process allows the model to understand the context and meaning of the input text.

QUERY: optimization techniques for training
Advanced RAG Answer:
 Based on the context, some optimization techniques for training include:

1. Adam optimizer: combines momentum and adaptive learning rates for efficient training.
2. Gradient descent: an optimization algorithm used to minimize loss functions.
3. Regularization techniques like dropout: help prevent overfitting in neural networks.

These techniques can improve the efficiency and effectiveness of training neural networks.

QUERY: what is tf-idf used for?
Advanced RAG Answer:
 TF-IDF is used in information retrieval.


In [34]:
print("| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |")
print("|------|-------------------|----------------------|------------|")

for q in queries:
    # Naïve
    naive_doc = naive_rag(q)
    
    # FULL Advanced RAG (correct pipeline)
    expanded = expand_query(q)
    
    all_docs = []
    seen = set()
    
    for eq in expanded:
        docs = retriever.retrieve(eq, top_k=5)
        
        for d in docs:
            if d["text"] not in seen:
                seen.add(d["text"])
                all_docs.append(d)
    
    reranked = rerank(q, all_docs, top_k=1)
    adv_doc = reranked[0]["text"]
    
    diff = "Yes" if naive_doc != adv_doc else "No"
    
    print(f"| {q} | {naive_doc[:40]}... | {adv_doc[:40]}... | {diff} |")

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |
|------|-------------------|----------------------|------------|
| how do transformers encode meaning? | Transformers use self-attention mechanis... | Transformers use self-attention mechanis... | No |
| optimization techniques for training | Gradient descent is an optimization algo... | Adam optimizer combines momentum and ada... | Yes |
| what is tf-idf used for? | TF-IDF stands for Term Frequency-Inverse... | TF-IDF stands for Term Frequency-Inverse... | No |


## Bonus 1

In [18]:
def weighted_rrf_fusion(query, alpha=0.5, k=60, top_k=5):
    tokenized_query = query.lower().split()
    
    # BM25
    bm25_scores = retriever.bm25.get_scores(tokenized_query)
    bm25_ranks = np.argsort(bm25_scores)[::-1]
    
    # SBERT
    query_emb = retriever.sbert.encode(query, convert_to_tensor=True)
    sbert_scores = util.cos_sim(query_emb, retriever.embeddings)[0].cpu().numpy()
    sbert_ranks = np.argsort(sbert_scores)[::-1]
    
    scores = {}
    
    for rank, doc_id in enumerate(bm25_ranks):
        scores.setdefault(doc_id, {"bm25_rank": None, "sbert_rank": None})
        scores[doc_id]["bm25_rank"] = rank + 1
    
    for rank, doc_id in enumerate(sbert_ranks):
        scores.setdefault(doc_id, {"bm25_rank": None, "sbert_rank": None})
        scores[doc_id]["sbert_rank"] = rank + 1
    
    results = []
    
    for doc_id, ranks in scores.items():
        score = 0
        
        if ranks["bm25_rank"]:
            score += alpha * (1 / (k + ranks["bm25_rank"]))
        if ranks["sbert_rank"]:
            score += (1 - alpha) * (1 / (k + ranks["sbert_rank"]))
        
        results.append((score, retriever.corpus[doc_id]))
    
    results.sort(reverse=True)
    return results[:top_k]

In [19]:
alphas = [0.3, 0.5, 0.7]

query = "what is tf-idf used for?"

for a in alphas:
    results = weighted_rrf_fusion(query, alpha=a, top_k=1)
    
    print(f"\nAlpha = {a}")
    print("Top doc:", results[0][1])


Alpha = 0.3
Top doc: TF-IDF stands for Term Frequency-Inverse Document Frequency and is used in information retrieval.

Alpha = 0.5
Top doc: TF-IDF stands for Term Frequency-Inverse Document Frequency and is used in information retrieval.

Alpha = 0.7
Top doc: TF-IDF stands for Term Frequency-Inverse Document Frequency and is used in information retrieval.


### Bonus 1 Observations

- Higher alpha (BM25-heavy) performs better for keyword queries like TF-IDF
- Lower alpha (semantic-heavy) helps vague queries
- Balanced alpha gives stable results

Conclusion:
Weighted RRF allows tuning retrieval behavior dynamically.

## Bonus 2

In [35]:
long_doc = """
Transformers are deep learning models that rely on attention mechanisms.
They process input data in parallel and capture long-range dependencies.
Training transformers requires optimization techniques like Adam and learning rate scheduling.
Regularization techniques such as dropout are used to prevent overfitting.
"""

In [36]:
def chunk_text(text, chunk_size):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

In [37]:
for size in [50, 100, 200]:
    chunks = chunk_text(long_doc, size)
    test_corpus = corpus + chunks
    
    temp_retriever = HybridRetriever(test_corpus)
    result = temp_retriever.retrieve("how do transformers train?", top_k=1)
    
    print(f"\nChunk size = {size}")
    print("Top doc:", result[0]["text"])


Chunk size = 50
Top doc: Transformers use self-attention mechanisms to encode relationships between words in a sequence.

Chunk size = 100
Top doc: Transformers use self-attention mechanisms to encode relationships between words in a sequence.

Chunk size = 200
Top doc: Transformers use self-attention mechanisms to encode relationships between words in a sequence.


### Bonus 2 Observations

(Note: A shorter document is used here for demonstration purposes.)

- Smaller chunks improve precision
- Larger chunks improve contextual understanding
- Medium chunks balance both

Conclusion:
Chunk size significantly impacts retrieval performance.

## Bonus 3

In [38]:
class ColBERTRetriever:
    def __init__(self, corpus):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.corpus = corpus
        
        # Token embeddings
        self.doc_tokens = [
            self.model.encode(doc.split(), convert_to_tensor=True)
            for doc in corpus
        ]

    def get_scores(self, query):
        query_tokens = self.model.encode(query.split(), convert_to_tensor=True)
        
        scores = []
        
        for doc_emb in self.doc_tokens:
            sim_matrix = util.cos_sim(query_tokens, doc_emb)
            max_sim = sim_matrix.max(dim=1).values.sum().item()
            scores.append(max_sim)
        
        return np.array(scores)

In [39]:
def hybrid_with_colbert(query, top_k=5, k=60):
    # ---- BM25 ----
    tokenized_query = query.lower().split()
    bm25_scores = retriever.bm25.get_scores(tokenized_query)
    bm25_ranks = np.argsort(bm25_scores)[::-1]
    
    # ---- SBERT ----
    query_emb = retriever.sbert.encode(query, convert_to_tensor=True)
    sbert_scores = util.cos_sim(query_emb, retriever.embeddings)[0].cpu().numpy()
    sbert_ranks = np.argsort(sbert_scores)[::-1]
    
    # ---- ColBERT ----
    colbert_scores = colbert.get_scores(query)
    colbert_ranks = np.argsort(colbert_scores)[::-1]
    
    scores = {}
    
    # Store ranks
    for rank, doc_id in enumerate(bm25_ranks):
        scores.setdefault(doc_id, {"bm25": None, "sbert": None, "colbert": None})
        scores[doc_id]["bm25"] = rank + 1
        
    for rank, doc_id in enumerate(sbert_ranks):
        scores.setdefault(doc_id, {"bm25": None, "sbert": None, "colbert": None})
        scores[doc_id]["sbert"] = rank + 1
        
    for rank, doc_id in enumerate(colbert_ranks):
        scores.setdefault(doc_id, {"bm25": None, "sbert": None, "colbert": None})
        scores[doc_id]["colbert"] = rank + 1
    
    # ---- RRF Fusion (3-way) ----
    results = []
    
    for doc_id, ranks in scores.items():
        rrf = 0
        
        if ranks["bm25"]:
            rrf += 1 / (k + ranks["bm25"])
        if ranks["sbert"]:
            rrf += 1 / (k + ranks["sbert"])
        if ranks["colbert"]:
            rrf += 1 / (k + ranks["colbert"])
        
        results.append((rrf, retriever.corpus[doc_id]))
    
    results.sort(reverse=True)
    return results[:top_k]

In [40]:
colbert = ColBERTRetriever(corpus)

In [41]:
query = "transformer attention mechanism"

results = hybrid_with_colbert(query, top_k=3)

print("3-Way Hybrid Results:")
for r in results:
    print("-", r[1])

3-Way Hybrid Results:
- The attention mechanism assigns weights to tokens based on relevance to a query.
- BERT is a transformer-based model pretrained using masked language modeling.
- Transformers use self-attention mechanisms to encode relationships between words in a sequence.


In [42]:
query = "what is tf-idf used for?"

print("Original Hybrid:")
print(retriever.retrieve(query, top_k=1)[0]["text"])

print("\nWith ColBERT:")
print(hybrid_with_colbert(query, top_k=1)[0][1])

Original Hybrid:
TF-IDF stands for Term Frequency-Inverse Document Frequency and is used in information retrieval.

With ColBERT:
TF-IDF stands for Term Frequency-Inverse Document Frequency and is used in information retrieval.


### Bonus 3 Observations

- ColBERT introduces token-level interaction using MaxSim scoring
- Unlike SBERT, it captures fine-grained term matching
- When fused with BM25 and SBERT via RRF, it improves ranking robustness
- Particularly useful for partial matches and technical queries

Conclusion:
Adding ColBERT as a third retriever enhances retrieval precision, though at higher computational cost.